# SEP Regression Onset Tool

This tool downloads SEP intensity time-series data and searches for an SEP event onset time using a regression method.

The user therefore selects a time interval in which the method looks for breakpoints in the data (using the logarithm of the intensities).

In [ ]:
import datetime as dt
import os
import pandas as pd
import regression_onset as reg
from regression_onset import select_data
from seppy.tools import Event

## Selecting the data source; choose either SEPpy or User defined

- `SEPpy` employs automatic data-loading capabilities online
- `User defined` accesses data from your local directory

In [ ]:
display(select_data.data_file)

### Provide file name of your own file (if used), otherwise select time interval to load data

In [ ]:
# This is the path to your data directory
path = f"{os.getcwd()}{os.sep}data"

# The name of your data file, if you're loading in your own data. 
filename = "solo_ept_sun_e.csv"

# To download (or load if files are locally present) SEPpy data, one needs to provide a time span.
# If you're not using SEPpy, this can be ignored.
# The format is (year, month, day)
start_date = dt.datetime(2022, 1, 20)
end_date = dt.datetime(2022, 1, 21)

if select_data._seppy_selected(select_data.data_file):
    import seppy.tools.widgets as w
    display(w.spacecraft_drop, w.sensor_drop, w.view_drop, w.species_drop)

## The next cell takes care of data loading. Just run it.

In [ ]:
if select_data._seppy_selected(select_data.data_file):
    # Initializes the SEPpy Event object
    seppy_data = Event(spacecraft=w.spacecraft_drop.value, sensor=w.sensor_drop.value, species=w.species_drop.value,
                         start_date=start_date, end_date=end_date, data_level="l2",
                         data_path=path, viewing=w.view_drop.value)
    
    # Exports the data to a pandas dataframe
    df = reg.externals.export_seppy_data(event=seppy_data)

else:
    # Uses pandas to_csv() to load in a local data file:
    df = pd.read_csv(f"{path}{os.sep}{filename}", parse_dates=True, index_col=0)

In [ ]:
# Check the dataframe
display(df)

## Use the quicklook-plot in the cell below to select a time interval in which the regression method is applied to find breakpoints:
#### The plot produced by `quicklook()` is an **interactive** plot.

Select a time interval that encompasses at least the background and peak of the event. Possible options to do this are: 

1) Clicking on the plot. A single click applies a selection UP TO the chosen time. Clicking again overwrites the previous click. This requires `selection = None`.
2) Use the `selection` parameter to select a time interval. `selection` can be either a single timestamp or a pair of timestamps given as strings, e.g., ["2025-03-19 12:00", "2025-03-20 18:00"]. A pair of timestamps defines a selection between the timestamps; a single timestamp defines selection up to that time from the start of the data file.

Vertical green line(s) on the plot indicate the selection.

In [ ]:
# Initializing the tool with input data
event = reg.Reg(data=df)

# Choose the channel (column name, see display(df) above)
channel = "E4"

# see explanation above for details 
# selection = ["2022-01-20 02:00", "2022-01-20 12:00"]
selection = None

# Display a quicklook plot of the input data (df).
# Apply the selection of data for the tool by 'selection' parameter or by clicking
# on the plot.
# The line magic 'ipympl' enables interactive mode
%matplotlib ipympl
event.quicklook(channel=channel, resample="5min", selection=selection)

## Regression analysis
### The method performs linear regression based on the logarithm of the intensity time profiles  (find_breakpoints() -method). Choose a number of breakpoints and run the method. It is the task of the user to identify, which breakpoint marks the onset of the event. Iterations using different numbers of breakpoints are recommended.

> fill_zeroes (bool) is a switch that applies filling out 0 count bins with a filler value f that satisfies the equation:
$$
\mu_{lg} = \frac{1}{N} \bigg( \sum_{i}^{N_{nz}} \lg(j_{i,nz}) + (N-N_{nz}) \lg(f) \bigg),
$$
> where $\mu_{lg}$ is the logarithm of the mean of the background, $N$ is the total number of data points in the background, $N_{nz}$ is the number of non-zero data in the background, $j_{i, nz}$ is a non-zero intensity measurement in the background and $\lg(\cdot)$ is the 10-base logarithm.
> 
> Setting fill_zeroes = False means 0 counts get ignored altogether.

In [ ]:
# The number of breakpoints to seek from the data selection
num_of_breaks = 3

# Fills zero counts with a filler falue f
fill_zeroes = True


# Time-averages the data to given cadence
resample = "1min"

# Boundaries of the time axis
xlim = ["2022-01-20 00:00", "2022-01-21 06:00"]

# Title for the figure (optional)
if select_data._seppy_selected(select_data.data_file):
    title = r"Solar Orbiter / EPT$^{\mathrm{sun}}$ ($0.0439 - 0.0467$) MeV electrons, 1 min data"
else:
    title = ''  # set own title for User defined file 

%matplotlib inline
results = event.find_breakpoints(channel=channel, breaks=num_of_breaks, fill_zeroes=fill_zeroes, xlim=xlim, title=title, resample=resample)

## Display the results of the regression analysis:
#### the results are a dictionary that contains the parameters of the fits (constant and slopes), the breakpoints and their 95% confidence intervals, the figure and its axes.

In [ ]:
display(results)

## Saving the figure:

In [ ]:
figure_name = "name_for_your_figure.png"

reg.externals.save_figure(results=results, name=figure_name)